In [1]:
import json, torch
from beir.datasets.data_loader import GenericDataLoader
from beir.retrieval import models
from beir.retrieval.evaluation import EvaluateRetrieval
from beir.retrieval.search.dense import DenseRetrievalExactSearch as DRES

def load_cs_queries(ds, field):
    # field is "ur_en" or "roman"; returns {qid: text}
    out = {}
    with open(f"queries/{ds}_cs.jsonl", encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            out[r["id"]] = r[field]
    return out

# single test: e5-base-v2 on scifact, ROMAN condition
ds = "scifact"
model_name = "intfloat/e5-base-v2"
corpus, _, qrels = GenericDataLoader(data_folder=f"data/{ds}").load(split="test")
queries_cs = load_cs_queries(ds, "roman")
print(f"{ds}: {len(corpus)} docs, {len(queries_cs)} roman queries")

beir_model = DRES(models.SentenceBERT(model_name), batch_size=32)
retriever = EvaluateRetrieval(beir_model, score_function="cos_sim")
results = retriever.retrieve(corpus, queries_cs)
ndcg, _map, recall, precision = retriever.evaluate(qrels, results, [10, 100])
mrr = retriever.evaluate_custom(qrels, results, [10], metric="mrr")

print("\n--- ROMAN condition ---")
print("nDCG@10  :", round(ndcg["NDCG@10"], 4))
print("Baseline EN nDCG@10 was ~0.7105")
print("Drop     :", round(0.7105 - ndcg["NDCG@10"], 4))

f:\Research Paper\AI2ML code-switching benchmark\venv\Lib\site-packages\beir\datasets\data_loader.py:8: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm
100%|██████████| 5183/5183 [00:00<00:00, 171508.75it/s]


scifact: 5183 docs, 300 roman queries


Batches: 100%|██████████| 162/162 [09:53<00:00,  3.67s/it]



--- ROMAN condition ---
nDCG@10  : 0.6209
Baseline EN nDCG@10 was ~0.7105
Drop     : 0.0896


In [2]:
import os, json, pandas as pd
from beir.datasets.data_loader import GenericDataLoader
from beir.retrieval import models
from beir.retrieval.evaluation import EvaluateRetrieval
from beir.retrieval.search.dense import DenseRetrievalExactSearch as DRES

os.makedirs("results", exist_ok=True)
out_path = "results/degradation.csv"

datasets = ["scifact", "nfcorpus"]
model_list = [
    "intfloat/e5-base-v2",
    "BAAI/bge-base-en-v1.5",
    "intfloat/multilingual-e5-base",
    "BAAI/bge-m3",
]
conditions = ["ur_en", "roman"]

# resume support: skip runs already in the csv
done = set()
if os.path.exists(out_path):
    prev = pd.read_csv(out_path)
    done = {(r.dataset, r.model, r.condition) for r in prev.itertuples()}
    print("Already done:", len(done), "runs")

def load_cs(ds, field):
    out = {}
    with open(f"queries/{ds}_cs.jsonl", encoding="utf-8") as f:
        for line in f:
            r = json.loads(line); out[r["id"]] = r[field]
    return out

rows = []
for ds in datasets:
    corpus, _, qrels = GenericDataLoader(data_folder=f"data/{ds}").load(split="test")
    for m in model_list:
        beir_model = None
        for cond in conditions:
            if (ds, m, cond) in done:
                print(f"skip {ds} | {m} | {cond}"); continue
            if beir_model is None:
                beir_model = DRES(models.SentenceBERT(m), batch_size=32)
                retriever = EvaluateRetrieval(beir_model, score_function="cos_sim")
            print(f"\n>>> {ds} | {m} | {cond}")
            q = load_cs(ds, cond)
            res = retriever.retrieve(corpus, q)
            ndcg, _map, recall, precision = retriever.evaluate(qrels, res, [10, 100])
            mrr = retriever.evaluate_custom(qrels, res, [10], metric="mrr")
            row = {"dataset": ds, "model": m, "condition": cond,
                    "nDCG@10": ndcg["NDCG@10"], "Recall@10": recall["Recall@10"],
                    "Recall@100": recall["Recall@100"], "MRR@10": mrr["MRR@10"]}
            rows.append(row)
            pd.DataFrame(rows).to_csv(
                out_path, mode="a", header=not os.path.exists(out_path), index=False)
            rows = []  # flush each time
            print(f"    nDCG@10 = {row['nDCG@10']:.4f}")

print("\nAll degradation runs complete -> results/degradation.csv")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6243.48it/s]



>>> scifact | intfloat/e5-base-v2 | ur_en


Batches: 100%|██████████| 162/162 [09:51<00:00,  3.65s/it]


    nDCG@10 = 0.6498

>>> scifact | intfloat/e5-base-v2 | roman


Batches: 100%|██████████| 162/162 [09:52<00:00,  3.66s/it]


...


>>> nfcorpus | BAAI/bge-m3 | ur_en


Batches: 100%|██████████| 114/114 [29:22<00:00, 15.46s/it] 


    nDCG@10 = 0.3116

>>> nfcorpus | BAAI/bge-m3 | roman


Batches: 100%|██████████| 114/114 [29:27<00:00, 15.51s/it] 


    nDCG@10 = 0.2983

All degradation runs complete -> results/degradation.csv


In [3]:
import json, pandas as pd
from beir.datasets.data_loader import GenericDataLoader
from beir.retrieval import models
from beir.retrieval.evaluation import EvaluateRetrieval
from beir.retrieval.search.dense import DenseRetrievalExactSearch as DRES

datasets = ["scifact", "nfcorpus"]
model_list = ["intfloat/e5-base-v2", "BAAI/bge-base-en-v1.5",
                "intfloat/multilingual-e5-base", "BAAI/bge-m3"]

def load_all(ds):
    rows = {}
    with open(f"queries/{ds}_cs.jsonl", encoding="utf-8") as f:
        for line in f:
            r = json.loads(line); rows[r["id"]] = r
    return rows

# identify which queries are ACTUALLY code-switched (differ from EN)
def cs_subset(rows, field):
    return {qid for qid, r in rows.items()
            if r[field].strip().lower() != r["EN"].strip().lower()}

out_rows = []
for ds in datasets:
    corpus, _, qrels = GenericDataLoader(data_folder=f"data/{ds}").load(split="test")
    rows = load_all(ds)
    for cond in ["ur_en", "roman"]:
        keep = cs_subset(rows, cond)
        # filter qrels + queries to code-switched subset only
        sub_qrels = {q: qrels[q] for q in keep if q in qrels}
        frac = len(keep) / len(rows)
        print(f"\n{ds} | {cond}: {len(keep)}/{len(rows)} queries are code-switched ({frac:.0%})")
        for m in model_list:
            beir_model = DRES(models.SentenceBERT(m), batch_size=32)
            retriever = EvaluateRetrieval(beir_model, score_function="cos_sim")
            q_cs = {qid: rows[qid][cond] for qid in keep}
            q_en = {qid: rows[qid]["EN"] for qid in keep}   # matched EN baseline on SAME subset
            res_cs = retriever.retrieve(corpus, q_cs)
            res_en = retriever.retrieve(corpus, q_en)
            nd_cs = retriever.evaluate(sub_qrels, res_cs, [10])[0]["NDCG@10"]
            nd_en = retriever.evaluate(sub_qrels, res_en, [10])[0]["NDCG@10"]
            drop = (nd_en - nd_cs) / nd_en * 100
            out_rows.append({"dataset": ds, "model": m, "condition": cond,
                                "n_cs": len(keep), "EN_subset": round(nd_en,4),
                                "CS": round(nd_cs,4), "drop_%": round(drop,1)})
            print(f"  {m}: EN={nd_en:.4f} -> {cond}={nd_cs:.4f}  drop={drop:.1f}%")

df = pd.DataFrame(out_rows)
df.to_csv("results/degradation_csonly.csv", index=False)
print("\nSaved -> results/degradation_csonly.csv")
df

100%|██████████| 5183/5183 [00:00<00:00, 161768.35it/s]



scifact | ur_en: 300/300 queries are code-switched (100%)


Batches: 100%|██████████| 162/162 [10:20<00:00,  3.83s/it]


  intfloat/e5-base-v2: EN=0.7105 -> ur_en=0.6498  drop=8.5%


Batches: 100%|██████████| 162/162 [10:19<00:00,  3.82s/it]


...

Batches: 100%|██████████| 114/114 [07:34<00:00,  3.98s/it]


  intfloat/multilingual-e5-base: EN=0.2996 -> roman=0.2537  drop=15.3%


Batches: 100%|██████████| 114/114 [30:38<00:00, 16.13s/it] 


  BAAI/bge-m3: EN=0.3048 -> roman=0.2679  drop=12.1%

Saved -> results/degradation_csonly.csv


,dataset,model,condition,n_cs,EN_subset,CS,drop_%
0,scifact,intfloat/e5-base-v2,ur_en,300,0.7105,0.6498,8.5
1,scifact,BAAI/bge-base-en-v1.5,ur_en,300,0.7376,0.5963,19.2
2,scifact,intfloat/multilingual-e5-base,ur_en,300,0.6887,0.6623,3.8
3,scifact,BAAI/bge-m3,ur_en,300,0.6437,0.6422,0.2
4,scifact,intfloat/e5-base-v2,roman,300,0.7105,0.6209,12.6
5,scifact,BAAI/bge-base-en-v1.5,roman,300,0.7376,0.6251,15.3
6,scifact,intfloat/multilingual-e5-base,roman,300,0.6887,0.6143,10.8
7,scifact,BAAI/bge-m3,roman,300,0.6437,0.5584,13.2
8,nfcorpus,intfloat/e5-base-v2,ur_en,136,0.3562,0.2916,18.1
9,nfcorpus,BAAI/bge-base-en-v1.5,ur_en,136,0.3779,0.2999,20.6
